# Project Index

| Module | Notebook |
|---|---|
| `check_demo` | [check_demo.ipynb](check_demo.ipynb) |
| `check_training` | [check_training.ipynb](check_training.ipynb) |
| `download_assets` | [download_assets.ipynb](download_assets.ipynb) |
| `model` | [model.ipynb](model.ipynb) |
| `prepare_data` | [prepare_data.ipynb](prepare_data.ipynb) |
| `test` | [test.ipynb](test.ipynb) |
| `test_nr` | [test_nr.ipynb](test_nr.ipynb) |
| `train` | [train.ipynb](train.ipynb) |
| `CIDNet` | [utils/CIDNet.ipynb](utils/CIDNet.ipynb) |
| `dataset` | [utils/dataset.ipynb](utils/dataset.ipynb) |
| `HVI_transform` | [utils/HVI_transform.ipynb](utils/HVI_transform.ipynb) |
| `HVI_transformer_utils` | [utils/HVI_transformer_utils.ipynb](utils/HVI_transformer_utils.ipynb) |
| `LCA` | [utils/LCA.ipynb](utils/LCA.ipynb) |
| `loss_funcs` | [utils/loss_funcs.ipynb](utils/loss_funcs.ipynb) |
| `metrics` | [utils/metrics.ipynb](utils/metrics.ipynb) |
| `niqe_utils` | [utils/niqe_utils.ipynb](utils/niqe_utils.ipynb) |
| `uranker_model` | [utils/uranker/uranker_model.ipynb](utils/uranker/uranker_model.ipynb) |
| `uranker_utils` | [utils/uranker/uranker_utils.ipynb](utils/uranker/uranker_utils.ipynb) |
| `utils` | [utils/utils.ipynb](utils/utils.ipynb) |
| `vgg19` | [utils/vgg19.ipynb](utils/vgg19.ipynb) |
| `test_file_writes` | [tests/test_file_writes.ipynb](tests/test_file_writes.ipynb) |
| `build_training_notebook` | [scripts/build_training_notebook.ipynb](scripts/build_training_notebook.ipynb) |
| `run_notebook` | [scripts/run_notebook.ipynb](scripts/run_notebook.ipynb) |
| `start_jupyter` | [scripts/start_jupyter.ipynb](scripts/start_jupyter.ipynb) |
| `build_project_index` | [scripts/build_project_index.ipynb](scripts/build_project_index.ipynb) |
| `convert_notebooks` | [scripts/convert_notebooks.ipynb](scripts/convert_notebooks.ipynb) |


## Source Integrity and Model Output Check

SHA-256 kiểm tra nguồn module; so sánh đầu ra mô hình trên cùng đầu vào và trọng số ngẫu nhiên.


In [1]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(ROOT / '.runtime') not in sys.path: sys.path.insert(0, str(ROOT / '.runtime'))
manifest = json.loads((ROOT / 'notebooks/source_manifest.json').read_text())
for name, record in manifest.items():
    notebook = json.loads((ROOT / record['notebook']).read_text(encoding='utf-8'))
    source = ''.join(''.join(c['source']) for c in notebook['cells']
                     if 'notebook-only' not in c.get('metadata', {}).get('tags', [])
                     and (c['cell_type'] == 'code' or c.get('metadata', {}).get('script_entrypoint')))
    assert hashlib.sha256(source.encode()).hexdigest() == record['sha256'], name
    compile(source, record['notebook'], 'exec')
print(f'PASS: {len(manifest)} module notebooks giữ nguyên nguồn và biên dịch hợp lệ.')


PASS: 26 module notebooks giữ nguyên nguồn và biên dịch hợp lệ.


In [2]:
import subprocess, torch
from model import myModel
torch.set_num_threads(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
reference_notebook = json.loads((ROOT / 'notebooks/model_reference.ipynb').read_text(encoding='utf-8'))
reference_source = ''.join(''.join(c['source']) for c in reference_notebook['cells'] if c['cell_type'] in ('code', 'raw'))
assert hashlib.sha256(reference_source.encode()).hexdigest() == reference_notebook['metadata']['source_sha256']
reference = {'__name__': 'original_reference'}
exec(compile(reference_source, 'original_model.py', 'exec'), reference)
report = {}
for device in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    torch.manual_seed(29)
    original = reference['myModel'](in_channels=3, feature_channels=32, use_white_balance=True).to(device).eval()
    converted = myModel(in_channels=3, feature_channels=32, use_white_balance=True).to(device).eval()
    converted.load_state_dict(original.state_dict())
    sample = torch.rand(1, 3, 32, 40, device=device)
    with torch.no_grad(): expected, actual = original(sample), converted(sample)
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    report[device] = float((actual - expected).abs().max())
    print(device.upper(), 'max absolute difference:', report[device])
    del original, converted, sample, expected, actual
torch.cuda.empty_cache()
(ROOT / 'logs/dev').mkdir(parents=True, exist_ok=True)
(ROOT / 'logs/dev/notebook_parity_report.json').write_text(json.dumps(report), encoding='utf-8')
print('PASS: logic đầu ra mô hình được giữ nguyên; không nạp checkpoint cũ.')


CPU max absolute difference: 0.0


CUDA max absolute difference: 0.0
PASS: logic đầu ra mô hình được giữ nguyên; không nạp checkpoint cũ.
